# ArcFace Migros Product Recognition - Training Notebook

**Dataset**: migros_dataset_v1
**Architecture**: Same as arcface_grocery_fixed_v2 (ResNet-34 + ArcFace + HAL)

**Key Adaptations:**
1. Uses migros_dataset_v1 annotation format (CSV with numeric class IDs)
2. Product ID to name mapping via SDP_Product&ID_Dataset_fix.csv
3. Hierarchical category structure (Icecek, Makarna, Pirinc, Sut, Krema_Ve_Sos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

# === UPDATE THIS PATH ===
DRIVE_ZIP_PATH = '/content/drive/MyDrive/arcface/migros_dataset_v1.zip'
EXTRACT_PATH = '/content/migros_dataset'

os.makedirs(EXTRACT_PATH, exist_ok=True)
print(f"Extracting...")
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print("Done!")

for item in os.listdir(EXTRACT_PATH):
    print(f"  {item}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import math
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
class Config:
    DATASET_ROOT = '/content/migros_dataset/migros_dataset_v1'
    TRAINING_DIR = os.path.join(DATASET_ROOT, 'Training')
    TESTING_DIR = os.path.join(DATASET_ROOT, 'Testing')
    ANNOTATIONS_DIR = os.path.join(DATASET_ROOT, 'Annotations')
    CHECKPOINT_DIR = '/content/drive/MyDrive/migros_checkpoints'

    # Model (same as arcface_grocery)
    FEATURE_DIM = 512
    ARCFACE_SCALE = 64
    ARCFACE_MARGIN = 0.5

    # HAL
    HAL_WEIGHT = 2.0

    # Training
    NUM_EPOCHS = 100
    BATCH_SIZE = 64
    LEARNING_RATE = 0.01
    MIN_LR = 0.0001
    WEIGHT_DECAY = 0.0001

    # Data
    IMAGE_SIZE = 224
    NUM_WORKERS = 4
    SAMPLES_PER_EPOCH = 50000

    SAVE_EVERY = 5
    EVAL_EVERY = 5
    SEED = 42

config = Config()
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
torch.backends.cudnn.benchmark = True

## Step 1: Load Product ID Mapping and Annotations

In [ ]:
def load_product_id_mapping(annotations_dir):
    """Load product ID to name mapping from SDP_Product&ID_Dataset_fix.csv"""
    mapping_file = os.path.join(annotations_dir, 'SDP_Product&ID_Dataset_fix.csv')
    id_to_name = {}

    with open(mapping_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                try:
                    product_id = int(parts[0])
                    product_name = parts[1].strip()
                    id_to_name[product_id] = product_name
                except ValueError:
                    continue

    print(f"Loaded {len(id_to_name)} product ID mappings")
    return id_to_name

id_to_name = load_product_id_mapping(config.ANNOTATIONS_DIR)
print(f"\nSample mappings:")
for i, (k, v) in enumerate(list(id_to_name.items())[:5]):
    print(f"  {k}: {v}")

In [ ]:
def parse_annotations(annotations_dir, testing_dir, id_to_name):
    """Parse _annotations.csv for test samples."""
    ann_file = os.path.join(annotations_dir, '_annotations.csv')
    all_samples = []
    referenced_product_ids = set()

    df = pd.read_csv(ann_file)
    print(f"Annotations columns: {df.columns.tolist()}")
    print(f"Total annotation rows: {len(df)}")

    for _, row in df.iterrows():
        filename = row['filename']
        product_id = int(row['class'])
        xmin, ymin = int(row['xmin']), int(row['ymin'])
        xmax, ymax = int(row['xmax']), int(row['ymax'])

        image_path = os.path.join(testing_dir, filename)

        if os.path.exists(image_path) and product_id in id_to_name:
            if xmax > xmin and ymax > ymin:
                all_samples.append({
                    'image_path': image_path,
                    'product_id': product_id,
                    'product_name': id_to_name[product_id],
                    'bbox': [xmin, ymin, xmax, ymax]
                })
                referenced_product_ids.add(product_id)

    print(f"Parsed {len(all_samples)} test samples")
    print(f"Found {len(referenced_product_ids)} unique product IDs in annotations")

    return all_samples, referenced_product_ids

test_samples_raw, annotation_product_ids = parse_annotations(
    config.ANNOTATIONS_DIR, config.TESTING_DIR, id_to_name
)

## Step 2: Build Training Set from Product Images

In [ ]:
def normalize_product_name(name):
    """Normalize product name for matching."""
    name = os.path.splitext(name)[0]
    name = name.lower().strip()
    return name

def build_training_set(training_dir, id_to_name, valid_product_ids=None):
    """Build training set from hierarchical folder structure."""
    image_paths = []
    class_to_idx = {}
    idx_to_class = {}
    class_to_category = {}
    categories = set()

    # Build name lookup for matching
    name_to_id = {}
    for pid, pname in id_to_name.items():
        norm_name = normalize_product_name(pname)
        name_to_id[norm_name] = pid

    # Find all training images
    all_images = []
    for root, dirs, files in os.walk(training_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, training_dir)
                all_images.append((full_path, rel_path, file))

    print(f"Total images in Training folder: {len(all_images)}")

    matched_count = 0

    for full_path, rel_path, filename in all_images:
        norm_filename = normalize_product_name(filename)

        # Find matching product ID
        product_id = None
        for norm_name, pid in name_to_id.items():
            if norm_name == norm_filename or norm_name in norm_filename or norm_filename in norm_name:
                product_id = pid
                break

        if product_id is None:
            continue

        if valid_product_ids is not None and product_id not in valid_product_ids:
            continue

        matched_count += 1
        product_name = id_to_name[product_id]

        # Get category from path
        path_parts = rel_path.replace('\\', '/').split('/')
        category = path_parts[0] if len(path_parts) >= 1 else 'Unknown'
        categories.add(category)

        class_name = f"{product_id}_{product_name}"

        if class_name not in class_to_idx:
            idx = len(class_to_idx)
            class_to_idx[class_name] = idx
            idx_to_class[idx] = class_name
            class_to_category[class_name] = category

        image_paths.append((full_path, class_to_idx[class_name], class_name))

    print(f"Matched {matched_count} training images")

    category_to_idx = {cat: idx for idx, cat in enumerate(sorted(categories))}

    return class_to_idx, idx_to_class, class_to_category, category_to_idx, image_paths

print("Building training set...")
class_to_idx, idx_to_class, class_to_category, category_to_idx, image_paths = build_training_set(
    config.TRAINING_DIR, id_to_name, valid_product_ids=annotation_product_ids
)

NUM_CLASSES = len(class_to_idx)
NUM_CATEGORIES = len(category_to_idx)

print(f"\n=== Dataset Summary ===")
print(f"Training classes: {NUM_CLASSES}")
print(f"Categories: {NUM_CATEGORIES} - {list(category_to_idx.keys())}")

## Step 3: Create Test Samples with Class Matching

In [ ]:
def build_test_samples(raw_samples, class_to_idx, id_to_name):
    """Match test samples to training class indices."""
    matched_samples = []
    unmatched = 0

    for sample in raw_samples:
        product_id = sample['product_id']
        product_name = id_to_name.get(product_id, '')
        class_name = f"{product_id}_{product_name}"

        if class_name in class_to_idx:
            matched_samples.append({
                'image_path': sample['image_path'],
                'bbox': sample['bbox'],
                'class_idx': class_to_idx[class_name],
                'class_name': class_name
            })
        else:
            unmatched += 1

    print(f"Matched {len(matched_samples)} test samples")
    if unmatched > 0:
        print(f"WARNING: {unmatched} samples could not be matched!")

    return matched_samples

test_samples = build_test_samples(test_samples_raw, class_to_idx, id_to_name)
print(f"\nFinal test samples: {len(test_samples)}")

## Step 4: Dataset Classes

In [ ]:
class MigrosTrainDataset(Dataset):
    def __init__(self, image_paths, class_to_category, category_to_idx, transform=None, epoch_size=50000):
        self.image_paths = image_paths
        self.class_to_category = class_to_category
        self.category_to_idx = category_to_idx
        self.transform = transform
        self.epoch_size = epoch_size
        self.cache = {}

        print(f"Caching {len(image_paths)} training images...")
        for path, class_idx, _ in tqdm(image_paths):
            try:
                self.cache[path] = Image.open(path).convert('RGB')
            except Exception as e:
                print(f"Failed to load: {path}")
        print(f"Cached {len(self.cache)} images")

    def __len__(self):
        return self.epoch_size

    def __getitem__(self, idx):
        real_idx = np.random.randint(0, len(self.image_paths))
        img_path, class_idx, class_name = self.image_paths[real_idx]

        if img_path in self.cache:
            image = self.cache[img_path].copy()
        else:
            image = Image.new('RGB', (224, 224), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        category = self.class_to_category.get(class_name, list(self.category_to_idx.keys())[0])
        cat_idx = self.category_to_idx[category]

        return image, class_idx, cat_idx


class MigrosTestDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['image_path']).convert('RGB')
            x1, y1, x2, y2 = sample['bbox']

            w, h = image.size
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            if x2 > x1 and y2 > y1:
                image = image.crop((x1, y1, x2, y2))
        except:
            image = Image.new('RGB', (224, 224), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        return image, sample['class_idx']

In [ ]:
# Transforms (same as paper)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Create datasets
train_dataset = MigrosTrainDataset(
    image_paths,
    class_to_category,
    category_to_idx,
    transform=train_transform,
    epoch_size=config.SAMPLES_PER_EPOCH
)

test_dataset = MigrosTestDataset(
    test_samples,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Test: {len(test_dataset)} samples, {len(test_loader)} batches")

## Step 5: Model Architecture (Same as arcface_grocery)

In [ ]:
class ArcFaceHead(nn.Module):
    """ArcFace with margin annealing."""
    def __init__(self, in_features, num_classes, scale=64.0, margin=0.5):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.num_classes = num_classes
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.current_margin = 0.0

    def set_margin(self, margin):
        self.current_margin = min(margin, self.margin)

    def forward(self, embeddings, labels):
        normalized_weights = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, normalized_weights)

        m = self.current_margin
        if m > 0:
            cos_m, sin_m = math.cos(m), math.sin(m)
            th = math.cos(math.pi - m)
            mm = math.sin(math.pi - m) * m

            sine = torch.sqrt(1.0 - cosine.pow(2).clamp(0, 1))
            phi = cosine * cos_m - sine * sin_m
            phi = torch.where(cosine > th, phi, cosine - mm)

            one_hot = F.one_hot(labels, self.num_classes).float()
            output = one_hot * phi + (1 - one_hot) * cosine
        else:
            output = cosine

        return output * self.scale

    def get_proxies(self):
        return F.normalize(self.weight, dim=1)

In [ ]:
class HALHead(nn.Module):
    """Hierarchical Auxiliary Loss head."""
    def __init__(self, class_to_category, category_to_idx, scale=64.0):
        super().__init__()
        self.scale = scale
        self.num_categories = len(category_to_idx)
        self.category_to_classes = defaultdict(list)

        for class_name, cat_name in class_to_category.items():
            cat_idx = category_to_idx[cat_name]
            self.category_to_classes[cat_idx].append(class_name)

    def forward(self, embeddings, cat_labels, class_proxies, class_to_idx):
        cat_proxies = []
        for cat_idx in range(self.num_categories):
            class_names = self.category_to_classes[cat_idx]
            class_indices = [class_to_idx[name] for name in class_names if name in class_to_idx]

            if class_indices:
                cat_proxy = class_proxies[class_indices].mean(dim=0)
                cat_proxy = F.normalize(cat_proxy, dim=0)
            else:
                cat_proxy = torch.zeros(class_proxies.size(1), device=class_proxies.device)

            cat_proxies.append(cat_proxy)

        cat_proxies = torch.stack(cat_proxies)
        logits = self.scale * F.linear(embeddings, cat_proxies)
        loss = F.cross_entropy(logits, cat_labels)

        return loss

In [ ]:
class ProductRecognitionModel(nn.Module):
    def __init__(self, num_classes, num_categories, class_to_category, category_to_idx,
                 class_to_idx, embedding_dim=512, scale=64.0, margin=0.5):
        super().__init__()

        resnet = models.resnet34(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        self.embedding = nn.Sequential(
            nn.Linear(512, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )

        self.arcface = ArcFaceHead(embedding_dim, num_classes, scale, margin)
        self.hal = HALHead(class_to_category, category_to_idx, scale)
        self.class_to_idx = class_to_idx
        self.embedding_dim = embedding_dim

    def get_embeddings(self, x):
        features = self.backbone(x).flatten(1)
        embeddings = self.embedding(features)
        return F.normalize(embeddings, dim=1)

    def forward(self, x, class_labels=None, cat_labels=None, compute_hal=True):
        embeddings = self.get_embeddings(x)

        if class_labels is not None:
            logits = self.arcface(embeddings, class_labels)
            hal_loss = None
            if compute_hal and cat_labels is not None:
                class_proxies = self.arcface.get_proxies()
                hal_loss = self.hal(embeddings, cat_labels, class_proxies, self.class_to_idx)

            return embeddings, logits, hal_loss

        return embeddings

In [ ]:
model = ProductRecognitionModel(
    num_classes=NUM_CLASSES,
    num_categories=NUM_CATEGORIES,
    class_to_category=class_to_category,
    category_to_idx=category_to_idx,
    class_to_idx=class_to_idx,
    embedding_dim=config.FEATURE_DIM,
    scale=config.ARCFACE_SCALE,
    margin=config.ARCFACE_MARGIN
).to(device)

print(f"Model: {NUM_CLASSES} classes, {NUM_CATEGORIES} categories")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 6: Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=config.LEARNING_RATE,
    momentum=0.9,
    weight_decay=config.WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=config.MIN_LR
)

scaler = GradScaler()

def get_margin(epoch, warmup=20):
    if epoch < warmup:
        return config.ARCFACE_MARGIN * epoch / warmup
    return config.ARCFACE_MARGIN

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'num_classes': NUM_CLASSES,
        'class_to_idx': class_to_idx,
        'idx_to_class': idx_to_class,
        'id_to_name': id_to_name,
    }, path)

def load_checkpoint(model, optimizer, scheduler, path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'], ckpt.get('metrics', {})

## Step 7: Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, device, hal_weight):
    model.train()
    total_loss, total_arc_loss, total_hal_loss = 0, 0, 0
    correct, total = 0, 0

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, class_labels, cat_labels in pbar:
        images = images.to(device, non_blocking=True)
        class_labels = class_labels.to(device, non_blocking=True)
        cat_labels = cat_labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            _, logits, hal_loss = model(images, class_labels, cat_labels, compute_hal=True)
            arc_loss = criterion(logits, class_labels)
            loss = arc_loss
            if hal_loss is not None:
                loss = loss + hal_weight * hal_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_arc_loss += arc_loss.item()
        if hal_loss is not None:
            total_hal_loss += hal_loss.item()

        _, predicted = logits.max(1)
        total += class_labels.size(0)
        correct += predicted.eq(class_labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.3f}', 'acc': f'{100*correct/total:.1f}%'})

    n = len(loader)
    return total_loss/n, total_arc_loss/n, total_hal_loss/n, 100*correct/total

In [ ]:
@torch.no_grad()
def build_reference_db(model, image_paths, transform, device):
    model.eval()
    embeddings, labels = [], []

    for path, class_idx, _ in image_paths:
        try:
            image = Image.open(path).convert('RGB')
            image = transform(image).unsqueeze(0).to(device)
            emb = model.get_embeddings(image).cpu().numpy()
            embeddings.append(emb)
            labels.append(class_idx)
        except:
            pass

    return np.vstack(embeddings), np.array(labels)


@torch.no_grad()
def evaluate(model, test_loader, ref_emb, ref_lbl, device):
    model.eval()

    query_emb, query_lbl = [], []
    for images, labels in test_loader:
        emb = model.get_embeddings(images.to(device)).cpu().numpy()
        query_emb.append(emb)
        query_lbl.append(labels.numpy())

    query_emb = np.vstack(query_emb)
    query_lbl = np.concatenate(query_lbl)

    sims = query_emb @ ref_emb.T
    sorted_idx = np.argsort(-sims, axis=1)

    results = {}
    for k in [1, 5, 10, 20]:
        hits = sum(q in ref_lbl[sorted_idx[i, :k]] for i, q in enumerate(query_lbl))
        results[f'HR@{k}'] = 100 * hits / len(query_lbl)

    aps = []
    for i, q in enumerate(query_lbl):
        relevant = ref_lbl[sorted_idx[i]] == q
        if relevant.sum() > 0:
            prec = np.cumsum(relevant) / np.arange(1, len(relevant) + 1)
            aps.append((prec * relevant).sum() / relevant.sum())
    results['mAP'] = 100 * np.mean(aps) if aps else 0

    return results

In [ ]:
history = {'loss': [], 'arc_loss': [], 'hal_loss': [], 'acc': [], 'lr': [], 'hr1': [], 'hr5': []}

start_epoch = 0
resume_path = os.path.join(config.CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(resume_path):
    start_epoch, _ = load_checkpoint(model, optimizer, scheduler, resume_path)
    print(f"Resumed from epoch {start_epoch}")

In [ ]:
print(f"\n{'='*70}")
print(f"Training: {config.NUM_EPOCHS} epochs | {NUM_CLASSES} classes | HAL weight: {config.HAL_WEIGHT}")
print(f"{'='*70}\n")

best_hr1 = 0

for epoch in range(start_epoch, config.NUM_EPOCHS):
    margin = get_margin(epoch)
    model.arcface.set_margin(margin)

    loss, arc_loss, hal_loss, acc = train_epoch(
        model, train_loader, optimizer, criterion, scaler, device, config.HAL_WEIGHT
    )

    lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['loss'].append(loss)
    history['arc_loss'].append(arc_loss)
    history['hal_loss'].append(hal_loss)
    history['acc'].append(acc)
    history['lr'].append(lr)

    print(f"Epoch {epoch+1:3d}/{config.NUM_EPOCHS} | "
          f"Loss: {loss:.4f} (Arc: {arc_loss:.4f}, HAL: {hal_loss:.4f}) | "
          f"Acc: {acc:.2f}% | LR: {lr:.5f} | M: {margin:.2f}")

    if (epoch + 1) % config.EVAL_EVERY == 0:
        print("  Evaluating...")
        ref_emb, ref_lbl = build_reference_db(model, image_paths, test_transform, device)
        results = evaluate(model, test_loader, ref_emb, ref_lbl, device)

        hr1, hr5 = results['HR@1'], results['HR@5']
        history['hr1'].append(hr1)
        history['hr5'].append(hr5)

        print(f"  HR@1: {hr1:.2f}% | HR@5: {hr5:.2f}% | HR@10: {results['HR@10']:.2f}% | mAP: {results['mAP']:.2f}%")

        if hr1 > best_hr1:
            best_hr1 = hr1
            save_checkpoint(model, optimizer, scheduler, epoch+1, results,
                          os.path.join(config.CHECKPOINT_DIR, 'best.pth'))
            print(f"  ★ New best!")

    if (epoch + 1) % config.SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, scheduler, epoch+1, {},
                       os.path.join(config.CHECKPOINT_DIR, f'epoch_{epoch+1}.pth'))

    save_checkpoint(model, optimizer, scheduler, epoch+1, {}, resume_path)

print(f"\n{'='*70}")
print(f"Training complete! Best HR@1: {best_hr1:.2f}%")
print(f"{'='*70}")

## Step 8: Final Evaluation

In [ ]:
best_path = os.path.join(config.CHECKPOINT_DIR, 'best.pth')
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best model from epoch {ckpt['epoch']}")

print("\nBuilding reference database...")
ref_emb, ref_lbl = build_reference_db(model, image_paths, test_transform, device)

print("Evaluating...")
results = evaluate(model, test_loader, ref_emb, ref_lbl, device)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
for k, v in results.items():
    print(f"{k}: {v:.2f}%")
print("="*50)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

epochs = range(1, len(history['loss']) + 1)

axes[0, 0].plot(epochs, history['loss'], label='Total')
axes[0, 0].plot(epochs, history['arc_loss'], label='ArcFace')
axes[0, 0].plot(epochs, history['hal_loss'], label='HAL')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(epochs, history['acc'])
axes[0, 1].set_title('Training Accuracy')
axes[0, 1].grid(True)

axes[1, 0].plot(epochs, history['lr'])
axes[1, 0].set_title('Learning Rate')
axes[1, 0].grid(True)

if history['hr1']:
    eval_epochs = range(config.EVAL_EVERY, len(history['loss'])+1, config.EVAL_EVERY)
    axes[1, 1].plot(eval_epochs, history['hr1'], 'o-', label='HR@1')
    axes[1, 1].plot(eval_epochs, history['hr5'], 's-', label='HR@5')
    axes[1, 1].set_title('Test Metrics')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(config.CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()